# Reasoning Segmentation - Hybrid

**팀명**: Team 7   
**Seed**: 428

Query 키워드(`all`, `every`, `both`, `each`)로 single/multi 자동 분기

# 1. 환경 설정

In [ ]:
# 라이브러리 import
import os
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from src.config import *
from src.utils import set_seed, rle_encode, load_image, create_directories, visualize_result
from src.models import ModelManager, Qwen3Model
from src.multi import is_multi_object_query
from pipeline.v3_hybrid import run_hybrid

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# 시드 설정
set_seed(SEED)
print(f"Seed fixed to: {SEED}")

In [ ]:
# 디렉토리 생성
if SAVE_INTERMEDIATE:
    create_directories(INTERMEDIATE_DIRS)
    print("Created intermediate directories")
else:
    print("SAVE_INTERMEDIATE=False, skipping intermediate directories")

Path(OUTPUT_DIR).mkdir(exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# 데이터 로드
test_df = pd.read_csv(TEST_CSV)

print(f"Test dataset loaded")
print(f"   - Total samples: {len(test_df)}")
print(f"   - Columns: {test_df.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(test_df.head(3))

# 2. 파라미터 설정

In [ ]:
# 파라미터 설정
TEST_MODE = False
NUM_TEST_SAMPLES = 5

VISUALIZE_RESULTS = True
NUM_VISUALIZE = 10
RANDOM_SAMPLES = True

print("Parameters configured:")
print(f"   - TEST_MODE: {TEST_MODE}")
print(f"   - NUM_TEST_SAMPLES: {NUM_TEST_SAMPLES}")
print(f"   - active_single: (see prompts.yaml)")
print(f"   - active_multi: (see prompts.yaml)")
print(f"   - VISUALIZE_RESULTS: {VISUALIZE_RESULTS}")

# 3. 실행

In [ ]:
# 모델 로딩
model_manager = ModelManager()
model_manager.load_sam()
print("SAM loaded!")

print("\nLoading Qwen3-VL-8B...")
qwen3 = Qwen3Model()
qwen3.load()
print("Qwen3-VL-8B loaded!")

In [ ]:
# 이미지 로드
if TEST_MODE:
    test_df_subset = test_df.head(NUM_TEST_SAMPLES)
    print(f"TEST MODE: Processing only {NUM_TEST_SAMPLES} samples")
else:
    test_df_subset = test_df
    print(f"FULL MODE: Processing all {len(test_df)} samples")

test_df_subset = test_df_subset.copy()
test_df_subset['is_multi'] = test_df_subset['query'].apply(is_multi_object_query)
print(f"   - Single-object queries: {(~test_df_subset['is_multi']).sum()}")
print(f"   - Multi-object queries: {test_df_subset['is_multi'].sum()}")

all_data = []

for idx, row in tqdm(test_df_subset.iterrows(), total=len(test_df_subset), desc="Loading images"):
    image_path = Path(DATASET_DIR) / "test" / f"{row['ID']}.jpg"
    image = load_image(str(image_path))

    all_data.append({
        'id': row['ID'],
        'image': image,
        'query': row['query'],
        'mllm_pred': None,
        'sam_mask': None,
    })

print(f"\nLoaded {len(all_data)} images")

In [ ]:
# Hybrid Pipeline (single/multi auto-routing)
run_hybrid(
    mllm_model=qwen3,
    sam_model=model_manager.sam,
    all_data=all_data,
    verbose=VERBOSE,
    unload_mllm=True,
)

In [ ]:
# 제출 파일 생성
submission = []

for item in tqdm(all_data, desc="RLE encoding"):
    rle_string = rle_encode(item['sam_mask'])
    submission.append({
        'ID': item['id'],
        'Label': rle_string
    })

submission_df = pd.DataFrame(submission)

expected_ids = test_df_subset['ID'].tolist()
actual_ids = submission_df['ID'].tolist()
assert actual_ids == expected_ids, "Order mismatch!"

if TEST_MODE:
    submission_path = Path(OUTPUT_DIR) / "submission_v3_test.csv"
else:
    submission_path = Path(OUTPUT_DIR) / SUBMISSION_FILE

submission_df.to_csv(submission_path, index=False)

print(f"\nSubmission saved: {submission_path}")
print(f"Total samples: {len(submission_df)}")

# 4. 테스트

In [ ]:
# 결과 시각화
if VISUALIZE_RESULTS and len(all_data) > 0:
    if RANDOM_SAMPLES:
        num_samples = min(NUM_VISUALIZE, len(all_data))
        sample_indices = np.random.choice(len(all_data), size=num_samples, replace=False)
        print(f"Visualizing {num_samples} random samples")
    else:
        sample_indices = list(range(min(NUM_VISUALIZE, len(all_data))))
        print(f"Visualizing first {len(sample_indices)} samples")

    for idx in sample_indices:
        item = all_data[idx]
        if item.get('is_multi') and item.get('mllm_pred', {}).get('bboxes'):
            print(f"[{item['id']}] MULTI — {len(item['mllm_pred']['bboxes'])} boxes")
        visualize_result(item)
else:
    print("Visualization skipped (VISUALIZE_RESULTS=False or no data)")